# Quantifying Mtb on a single-cell level across large tissue slices

First work on a single example to determine thresholds then iterate over all slices.

In [1]:
import zarr
import glob
import napari
import dask.array as da
from pathlib import Path

In [2]:
zarr_address = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_1/zarr/*zarr')[0]

In [3]:
print(zarr_address)

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_1/zarr/20250815_40X_TimerMtb_BP_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250815_5375.zarr


In [32]:
viewer = napari.Viewer()
viewer.open(Path(zarr_address,'0'), channel_axis = 1)   # napari-ome-zarr plugin


# # --- show in napari ---
# viewer = napari.Viewer()
# viewer.add_image(img, name="image")
# viewer.add_labels(lbl, name="ground_truth_mtb")
# napari.run()


[<Image layer '0' at 0x7f48ceca3890>,
 <Image layer '0 [1]' at 0x7f48ced1f190>,
 <Image layer '0 [2]' at 0x7f48c2ccbcd0>]

In [36]:
viewer.open(Path(zarr_address, 'labels', 'ground_truth_mtb'))


/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


[<Image layer 'ground_truth_mtb' at 0x7f498c1997d0>]

In [4]:

# --- load image (full-res level) ---
img = da.from_zarr(f"{zarr_address}/0/0")    # adjust path if your top-level differs

# --- load label ---
lbl = da.from_zarr(f"{zarr_address}/labels/ground_truth_mtb/0")


In [4]:
viewer = napari.Viewer(title = 'testing mask output')

In [9]:
viewer.title = 'creating heatmap'

In [6]:
viewer.add_labels(lbl)

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


<Labels layer 'lbl' at 0x771d6813e4d0>

In [5]:
img

dask.array<from-zarr, shape=(1, 3, 11, 31334, 49997), dtype=>u2, chunksize=(1, 1, 1, 1024, 1024), chunktype=numpy.ndarray>

In [15]:
%%time
max_proj = img.max(axis=2).compute()

CPU times: user 11min 7s, sys: 3min 48s, total: 14min 55s
Wall time: 21min 53s


In [16]:

viewer.add_image(max_proj, channel_axis=1)

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


[<Image layer 'Image' at 0x771d6bd387d0>,
 <Image layer 'Image [1]' at 0x771d6bd09790>,
 <Image layer 'Image [2]' at 0x771d6bec4b10>]

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


In [11]:
lbl

dask.array<from-zarr, shape=(41702, 58291), dtype=uint8, chunksize=(1304, 1822), chunktype=numpy.ndarray>

# Load previously calculated results

In [6]:
import pandas as pd

In [9]:
df = pd.read_pickle('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/sc_results_w_bb.pkl')

In [10]:
df['timer_ratio'] = df['mean_intensity_ch1']/df['mean_intensity_ch2']

In [15]:
df['x'] = (df['bbox_rmax'] - df['bbox_rmin'])//2 + df['bbox_rmin']		
df['y'] = (df['bbox_cmax'] - df['bbox_cmin'])//2 + df['bbox_cmin']		

In [16]:
df

,zarr_address,region_id,mean_intensity_ch1,mean_intensity_ch2,area,bbox_rmin,bbox_rmax,bbox_cmin,bbox_cmax,timer_ratio,x,y
0,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...,1,0.000000,0.000000,23.0,1267,1279,32515,32543,NaN,1273,32529
1,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...,2,0.000000,0.000000,39.0,2019,2038,21217,21233,NaN,2028,21225
2,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...,3,120.315068,110.698630,146.0,2433,2459,22886,22916,1.086870,2446,22901
3,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...,4,883.984615,250.476923,65.0,2496,2524,32883,32908,3.529206,2510,32895
4,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...,5,0.000000,0.000000,91.0,2500,2521,9392,9419,NaN,2510,9405
...,...,...,...,...,...,...,...,...,...,...,...,...
26999,20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_Ti...,239,137.860759,158.700422,237.0,36480,36503,21175,21213,0.868686,36491,21194
27000,20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_Ti...,240,109.032258,114.903226,62.0,36519,36544,16639,16654,0.948905,36531,16646
27001,20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_Ti...,241,302.518325,407.848168,191.0,36544,36574,20714,20741,0.741743,36559,20727
27002,20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_Ti...,242,110.038760,117.930233,129.0,36584,36601,14265,14296,0.933084,36592,14280


In [13]:
for i, group in df.groupby('zarr_address'):
    print(group['zarr_address'].iloc[0])
    print(len(group))
    

20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375.zarr
23226
20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5550.zarr
309
20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250902_5555.zarr
243
20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5551.zarr
1912
20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5552.zarr
396
20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5553.zarr
455
20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5555.zarr
354
20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.zarr
109


# Need to fix the quantification in 5375 first

In [11]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np # Used only for dummy data if file fails

In [9]:
df

,zarr_address,region_id,mean_intensity_ch1,mean_intensity_ch2,area,bbox_rmin,bbox_rmax,bbox_cmin,bbox_cmax,timer_ratio,x,y
0,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...,1,0.000000,0.000000,23.0,1267,1279,32515,32543,NaN,1273,32529
1,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...,2,0.000000,0.000000,39.0,2019,2038,21217,21233,NaN,2028,21225
2,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...,3,120.315068,110.698630,146.0,2433,2459,22886,22916,1.086870,2446,22901
3,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...,4,883.984615,250.476923,65.0,2496,2524,32883,32908,3.529206,2510,32895
4,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...,5,0.000000,0.000000,91.0,2500,2521,9392,9419,NaN,2510,9405
...,...,...,...,...,...,...,...,...,...,...,...,...
26999,20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_Ti...,239,137.860759,158.700422,237.0,36480,36503,21175,21213,0.868686,36491,21194
27000,20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_Ti...,240,109.032258,114.903226,62.0,36519,36544,16639,16654,0.948905,36531,16646
27001,20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_Ti...,241,302.518325,407.848168,191.0,36544,36574,20714,20741,0.741743,36559,20727
27002,20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_Ti...,242,110.038760,117.930233,129.0,36584,36601,14265,14296,0.933084,36592,14280


In [15]:
for i, group in df.groupby('zarr_address'):
    print(i, group)
    break

20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375.zarr                                             zarr_address  region_id  \
0      20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...          1   
1      20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...          2   
2      20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...          3   
3      20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...          4   
4      20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...          5   
...                                                  ...        ...   
26756  20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...       6460   
26757  20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...       6461   
26758  20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...       6462   
26759  20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...       6463   
26760  20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...       6464   

       mean_intensity_ch1  mean_intensity_ch2   area  bb

In [21]:
expt_ID = i.split('_')[-1].split('.')[0]

'5375'